# Decision Tree - Classification - Quotation Data

In [4]:
# Load data
import pandas as pd

quotation_data_path = r"C:\Users\Phong\OneDrive - ICB Construction\Phong\data\Python_ETL\DS\ML_Models\data\Quotation Data.xlsx"
# Date_Columns = ["Due_Date", "Date_Sent"]

# This creates: {"Due_Date": str, "Date_Sent": str}
# dtype_mapping = {col: str for col in Date_Columns}

# quotation_data_df = pd.read_excel(quotation_data_path, dtype=dtype_mapping)
quotation_data_df = pd.read_excel(quotation_data_path)
#quotation_data_df.describe()
#uotation_data_df.head(2)

In [5]:
# Lib Imports
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, precision_recall_curve
from sklearn.calibration import calibration_curve
from joblib import dump

In [11]:
# Feature Engineering
# - Date and Value columns transformed - Integer only
# - Priced_By feature transformed
 
# ================================================================
# Custom Date Feature Engineering
# ================================================================
class DateFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, date_column="Date"):
        self.date_column = date_column

    def fit(self, X, y=None):
        dates = pd.to_datetime(
            X[self.date_column],
            errors="coerce"
        )
        self.median_date_ = dates.dropna().median()
        return self

    def transform(self, X):
        X = X.copy()
        X[self.date_column] = pd.to_datetime(
            X[self.date_column],
            errors="coerce"
        )

        # Missing Indicator
        X["DateMissing"] = X[self.date_column].isna().astype(int)

        # Median Imputation
        X[self.date_column] = X[self.date_column].fillna(
            self.median_date_
        )

        # Date Features
        X["Date_Year"] = X[self.date_column].dt.year
        X["Date_Month"] = X[self.date_column].dt.month
        X["Date_DayOfWeek"] = X[self.date_column].dt.dayofweek

        X.drop(columns=self.date_column, inplace=True)
        return X

# ================================================================
# Numeric Feature Engineering
# ================================================================
class NumericFeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_value_ = X["Value"].median()
        return self
    
    def transform(self, X):
        X = X.copy()
        X["Value"] = X["Value"].fillna(self.median_value_)
        X["Value"] = np.log1p(X["Value"])
        return X

# ================================================================
# Estimator Feature Engineering
# ================================================================
class EstimatorFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, column="Priced_By"):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Number of estimators
        X["EstimatorCount"] = (
            X[self.column]
            .str.split("/")
            .apply(len)
        )

        # If the value is "MISSING", treat it as 0 estimators
        X.loc[
            X[self.column].str.upper() == "MISSING",
            "EstimatorCount"
        ] = 0

        X["EstimatorCount"] = X["EstimatorCount"].astype(int)

        # Missing Estimator flag
        X["EstimatorMissing"] = (
            X[self.column]
            .str.upper()
            .eq("MISSING")
        ).astype(int)

        return X

In [16]:
# Feature Lists and Data Split
# - Use TimeSeriesSplit to handle the Date_Year feature 
binary_features = [
    "Timber_RW","RC_Pile","Steel_Beam","Sheetpile","Anchor","Block",
    "Shotcrete","Capping_Beam","Earthwork","Concrete_Slab","Precast",
    "Culvert","Slip_Repair","Soil_Nail","Rock_RW","Bridge","Concrete",
    "Design_and_Build","Budget","Drill_Only","Labour_Only",
    "Driven_Pile","Palisade","Boardwalk","Soldier","Insitu",
    "Barrier","Noise_RW","Base","Casing","Crib",
    "DayWork","Flood_Repair","Micro_Pile","Reno",
    "Temp_RW","Other",
    "DateMissing",
    # "MultipleEstimators",
    "EstimatorMissing"
]

numeric_features = [
    "Value",
    "No_Wall_Types",
    "Date_Year",
    "Date_Month",
    "Date_DayOfWeek",
    "EstimatorCount"
]

categorical_features = [
    "Priced_By",
    # "Contact_Clean",
    "Suburb",
    "Client_Clean"
]

# Raw Features ONLY
raw_features = [
    "Date", "Value", "No_Wall_Types", "Priced_By", # "Contact_Clean",
    "Suburb", "Client_Clean", "Timber_RW","RC_Pile","Steel_Beam",
    "Sheetpile","Anchor","Block", "Shotcrete","Capping_Beam","Earthwork",
    "Concrete_Slab","Precast", "Culvert","Slip_Repair","Soil_Nail",
    "Rock_RW","Bridge","Concrete", "Design_and_Build","Budget",
    "Drill_Only","Labour_Only", "Driven_Pile","Palisade","Boardwalk",
    "Soldier","Insitu", "Barrier","Noise_RW","Base","Casing","Crib",
    "DayWork","Flood_Repair","Micro_Pile","Reno", "Temp_RW","Other"
]

target = "Success"

# ================================================================
# Train Test Split
# ================================================================
# Sort the Date
quotation_data_df["Date"] = pd.to_datetime(
    quotation_data_df["Date"],
    errors="coerce"
)

quotation_data_df = (
    quotation_data_df
    .sort_values("Date")
    .reset_index(drop=True)
)

X = quotation_data_df[raw_features]
y = quotation_data_df[target]

tscv = TimeSeriesSplit(n_splits=5)

split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

print(f"Training Samples : {len(X_train)}")
print(f"Testing Samples  : {len(X_test)}")

Training Samples : 3696
Testing Samples  : 924


In [19]:
# Preprocessing
# ================================================================
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
    # ("scaler", StandardScaler()) dont need scaling for tree based model
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("bin", "passthrough", binary_features)
    ]
)

In [18]:
# Decision Tree Pipeline
# ================================================================
decision_tree_pipeline = Pipeline([
    ("date_features", DateFeatureTransformer()),
    ("estimator_features", EstimatorFeatureTransformer(
            column="Priced_By"
        )),
    ("numeric_features", NumericFeatureTransformer()),
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        random_state=42
    ))
])

# ================================================================
# Hyperparameter Search
# ================================================================
param_grid = {
    "classifier__criterion": ["gini", "entropy", "log_loss"],
    "classifier__max_depth": [None, 3, 5, 7, 10, 15, 20],
    "classifier__min_samples_split": [2, 5, 10, 20, 50],
    "classifier__min_samples_leaf": [1, 2, 5, 10, 20],
    "classifier__max_features": [None, "sqrt", "log2"],
    "classifier__class_weight": [None, "balanced"],
    "classifier__ccp_alpha": [0.0, 0.0001, 0.0005, 0.001, 0.005, 0.01]
}

random_search = RandomizedSearchCV(
    estimator=decision_tree_pipeline,
    param_distributions=param_grid,
    n_iter=150,
    cv=tscv,
    scoring="balanced_accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# ================================================================
# Train
# ================================================================
random_search.fit(X_train, y_train)

print("\nBest Parameters")
print(random_search.best_params_)

print(f"\nBest CV Balanced Accuracy : {random_search.best_score_:.4f}")

rt_best_model_v1 = random_search.best_estimator_
# 1. Generate predictions from your latest model execution
y_pred = rt_best_model_v1.predict(X_test)

# 2. Extract and print the Classification Report
print("\n📊 Final Model Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Loss (0)', 'Profit (1)']))
print("=" * 60)

y_proba = rt_best_model_v1.predict_proba(X_test)[:, 1]
auc_roc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC: {auc_roc:.4f}")

# Save Trained Model
dump(rt_best_model_v1, "RandomTreeClassification_V1.joblib")
print("Model saved.")


Fitting 5 folds for each of 150 candidates, totalling 750 fits

Best Parameters
{'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 1, 'classifier__max_features': None, 'classifier__max_depth': 7, 'classifier__criterion': 'gini', 'classifier__class_weight': 'balanced', 'classifier__ccp_alpha': 0.001}

Best CV Balanced Accuracy : 0.6740

📊 Final Model Classification Report:
              precision    recall  f1-score   support

    Loss (0)       0.83      0.89      0.86       749
  Profit (1)       0.31      0.22      0.25       175

    accuracy                           0.76       924
   macro avg       0.57      0.55      0.56       924
weighted avg       0.73      0.76      0.74       924

AUC-ROC: 0.6923
Model saved.


In [22]:
# Feature Importance
feature_names = rt_best_model_v1.named_steps[
    "preprocessor"
].get_feature_names_out()

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rt_best_model_v1.named_steps[
        "classifier"
    ].feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)
print(importance.head(30))

                                  Feature  Importance
2                          num__Date_Year    0.603587
953                 bin__EstimatorMissing    0.221761
0                              num__Value    0.034403
917                       bin__Steel_Beam    0.023898
946                          bin__DayWork    0.019993
118                 cat__Suburb_mahurangi    0.018838
585           cat__Client_Clean_marchcato    0.014460
918                        bin__Sheetpile    0.012573
936                      bin__Driven_Pile    0.012122
167                    cat__Suburb_pokeno    0.011369
261               cat__Client_Clean_aspec    0.009630
14                    cat__Priced_By_GARY    0.008891
492             cat__Client_Clean_higgins    0.008475
7                       cat__Priced_By_CB    0.000000
938                        bin__Boardwalk    0.000000
11                    cat__Priced_By_DEAN    0.000000
12                    cat__Priced_By_DEEP    0.000000
13                  cat__Pri